## Importing the dataset

In [ ]:
from datasets import load_dataset
import re

# 1. Load the raw dataset
print("Loading Mohammed-Altaf/medical-instruction-120k...")
raw_dataset = load_dataset("Mohammed-Altaf/medical-instruction-120k")

def advanced_medical_cleaning(example):
    text = example['Conversation']

    text = re.sub(r"The conversation between human and AI assistant\.\s*", "", text, flags=re.IGNORECASE)

    text = re.sub(r"my name is \w+", "my name is [PATIENT]", text, flags=re.IGNORECASE)
    text = re.sub(r"i am \d+ years? old", "i am [AGE] years old", text, flags=re.IGNORECASE)

    fluff_phrases = [
        r"Hi,? Hope this message finds you in good health\.",
        r"I hope I have answered your query\.",
        r"If you have further doubts, I would be happy to help you\.",
        r"Take care, God bless\.",
        r"Thanks for consulting .* on HealthCareMagic\."
    ]
    for phrase in fluff_phrases:
        text = re.sub(phrase, "", text, flags=re.IGNORECASE)

    # 4. Normalize Whitespace
    text = re.sub(r'\s+', ' ', text) # Replace all newlines/tabs with single space
    text = text.replace("[|Human|]", "\n[|Human|] ").replace("[|AI|]", "\n[|AI|] ")

    return {"text": text.strip() + " <|endoftext|>"}

# Apply to your dataset
dataset = raw_dataset.map(advanced_medical_cleaning, remove_columns=raw_dataset['train'].column_names)


## Tokenization

In [ ]:
from transformers import PreTrainedTokenizerFast
from tokenizers import ByteLevelBPETokenizer

# 1. Load the raw BPE tokenizer first
raw_tokenizer = ByteLevelBPETokenizer(
    "/content/sample_data/vocab_50m (1).json",
    "/content/sample_data/merges_50m (1).txt",
)

# 2. Save it as a single 'tokenizer.json' file (This is what Transformers expects)
raw_tokenizer.save("tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer.json",
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
    unk_token="<|unknown|>",
    pad_token="<|padding|>",
    mask_token="<|mask|>"
)



In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding=False
    )

print("⚡ Tokenizing 112k samples...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=4
)


In [ ]:
import numpy as np
import os
import random

# --- CONFIGURATION ---
TRAIN_BIN = "train.bin"
VAL_BIN = "val.bin"
TRAIN_RATIO = 0.98

# 1. Extract and convert to LIST (This fixes the TypeError)
print("📦 Converting dataset column to list...")
all_ids = list(tokenized_dataset['train']['input_ids'])

# 2. Shuffle
print("🔀 Shuffling samples...")
random.seed(42)
random.shuffle(all_ids)

# 3. Split Data
split_idx = int(len(all_ids) * TRAIN_RATIO)
train_ids_list = all_ids[:split_idx]
val_ids_list = all_ids[split_idx:]

def save_to_bin(filename, data_list):

    print(f"🔨 Flattening and saving {filename}...")
    flat_data = np.concatenate([np.array(x, dtype=np.uint16) for x in data_list])

    # Export to binary
    with open(filename, 'wb') as f:
        f.write(flat_data.tobytes())

    return len(flat_data)

# 4. Execute Export
train_tokens = save_to_bin(TRAIN_BIN, train_ids_list)
val_tokens = save_to_bin(VAL_BIN, val_ids_list)

# 5. Final Report
print("\n" + "="*30)
print(f"✅ SUCCESS: {TRAIN_BIN} and {VAL_BIN} created.")
print(f"📊 Total tokens: {train_tokens + val_tokens:,}")
print("="*30)

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# Define the folder path containing vocab.json and merges.txt
vocab_path = "/kaggle/input/datasets/aman0419/vital-tokenizer-50m/vocab_50m.json"
merges_path = "/kaggle/input/datasets/aman0419/vital-tokenizer-50m/merges_50m.txt"

# Initialize directly
tokenizer = ByteLevelBPETokenizer(
    vocab=vocab_path,
    merges=merges_path
)


print("✅ Tokenizer loaded successfully!")

In [ ]:

import json

def clean_notebook_metadata(filepath):
    with open(filepath, 'r') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata that causes KeyError: 'state'
    if 'widgets' in nb.get('metadata', {}):
        nb['metadata'].pop('widgets')
        print("✅ Problematic widget metadata removed.")


In [ ]:
import torch
import numpy as np
import os

# --- CONFIGURATION ---
batch_size = 64
block_size = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def find_file(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    # If not in input, check the current working directory
    if os.path.exists(filename):
        return filename
    return None

train_path = find_file('sft_train.bin')
val_path = find_file('sft_test.bin')

if train_path and val_path:
    print(f"✅ Found files at: {train_path}")
    train_data = np.memmap(train_path, dtype=np.uint16, mode='r')
    val_data = np.memmap(val_path, dtype=np.uint16, mode='r')
else:
    raise FileNotFoundError("Could not find sft_train.bin or sft_test.bin in /kaggle/input or /kaggle/working")
print(f"📂 Loading memmaps from {train_path} and {val_path}...")

# Check if files exist to avoid crashing later
if not os.path.exists(train_path) or not os.path.exists(val_path):
    print("❌ Error: Binary files not found! Did you run the 'prepare_bin_files.py' script?")
else:
    # Load memory-mapped files (Instant access, low RAM usage)
    train_data = np.memmap(train_path, dtype=np.uint16, mode='r')
    val_data = np.memmap(val_path, dtype=np.uint16, mode='r')
    print(f"✅ Loaded! Train tokens: {len(train_data):,}, Val tokens: {len(val_data):,}")

# --- 2. GET_BATCH FUNCTION ---
def get_batch(split):
    # Select the correct dataset
    data = train_data if split == 'train' else val_data

    # Generate random starting indices
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Stack inputs (x) and targets (y)
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # Move to GPU (Asynchronous is faster)
    if device == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

## Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps

    def forward(self, x):
        return F.layer_norm(x, x.shape[-1:], self.weight, self.bias, self.eps)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = 4 * config.n_embd

        # w1: Gate Projection
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # w2: Value Projection
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # c_proj: Output Projection (Down projection)
        self.c_proj = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # SwiGLU Logic: (SiLU(Gate) * Value) -> Projection
        x = F.silu(self.w1(x)) * self.w2(x)

        # 4. Output projection
        return self.dropout(self.c_proj(x))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class SLMConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
config = SLMConfig(
    vocab_size=16384,   # Found in logs (torch.Size([16384, 512]))
    block_size=256,
    n_layer=10,         # Matches your Stage 1
    n_head=8,           # Matches your Stage 1 (512 / 64 = 8)
    n_embd=512,         # Matches your Stage 1
    dropout=0.1,
    bias=True
)

model = SLM(config).to(device)

In [ ]:

import torch
from contextlib import nullcontext

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    ptdtype = torch.float16 if device_type == 'cuda' else torch.float32
    inner_ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) if device_type == 'cuda' else nullcontext()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with inner_ctx: # Use the local inner_ctx
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Model Training

In [ ]:
import time
import math
from contextlib import nullcontext
import torch
import torch.nn as nn

# --- 1. INITIALIZE TRACKING LISTS ---
train_loss_list = []
val_loss_list = []
lr_list = []

# --- 2. SFT HYPERPARAMETERS ---
max_iters = 4500         
warmup_steps = 200      
learning_rate = 2e-5     
min_lr = 2e-6            
eval_interval = 250
eval_iters = 100
weight_decay = 0.1
grad_clip = 1.0
batch_size = 64         
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 3. MODEL INITIALIZATION & WEIGHT LOADING ---
model = SLM(config).to(device)

# Path to  Stage 1 (Pre-training) weights on Kaggle
WEIGHTS_PATH = "/kaggle/input/models/aman0419/vitallm-50m-sft/pytorch/default/1/vital_lm_50m_weights.pt"

print(f"🔄 Loading Stage 1 Weights: {WEIGHTS_PATH}")
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
state_dict = checkpoint.get('model_state_dict', checkpoint)

# Professional Key Cleaner: Strips 'module.' prefix from DataParallel
unwanted_prefix = 'module.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
print("✅ Weights successfully loaded for SFT Alignment.")

# --- 4. OPTIMIZER & MIXED PRECISION SETUP ---
dtype = 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type='cuda', dtype=ptdtype)
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# Separate decay/no-decay params
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': weight_decay},
    {'params': nodecay_params, 'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-9)

# --- 5. SCHEDULER & BATCH UTILS ---
def get_lr(it):
    if it < warmup_steps:
        return learning_rate * (it + 1) / (warmup_steps + 1)
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_iters - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# --- 6. SFT TRAINING LOOP ---
model.train()
t0 = time.time()
best_val_loss = float('inf')

print(f"🚀 Starting SFT for {max_iters} iterations on {device}...")

for iter in range(max_iters):

    # A. Set Learning Rate
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lr_list.append(lr)

    # B. Evaluation & Early Stopping
    if iter % eval_interval == 0 and iter > 0:
        model.eval()
        with torch.no_grad():
            losses = estimate_loss()
        model.train()

        train_loss_list.append(losses['train'])
        val_loss_list.append(losses['val'])

        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.2e}")

        # Early Stopping Logic: If Val Loss diverges significantly from best
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), "VitalLM_SFT_Best.pt")
        elif iter > 1500 and losses['val'] > (best_val_loss * 1.05):
            print("🛑 Overfit detected (Val Loss Divergence). Ending SFT early.")
            break

    # C. Training Step
    xb, yb = get_batch('train') # Ensure get_batch handles sft_train.bin

    with ctx:
        logits, loss = model(xb, yb)

    # Backward Pass with Scaler
    scaler.scale(loss).backward()

    # Clip Gradients
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    # Optimizer Step
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # Speed Logging
    if iter % 100 == 0:
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        tps = (batch_size * config.block_size * 100) / dt
        print(f"iter {iter} | loss {loss.item():.4f} | speed {tps:.0f} tok/s")

print("\n✅ Training Complete.")
torch.save(model.state_dict(), "VitalLM_SFT_Final.pt")

## Training Performance Analytics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_sft_performance(train_losses, val_losses, learning_rates, eval_interval):
   
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

    # Generate X-axis (iterations where eval happened)
    iters = [i * eval_interval for i in range(1, len(val_losses) + 1)]

    # 1. LOSS CURVE (Training vs Validation)
    ax1.plot(iters, train_losses, label='Est. Train Loss', color='#3498db', linewidth=2, marker='o', markersize=4)
    ax1.plot(iters, val_losses, label='Val Loss', color='#e74c3c', linewidth=2, marker='s', markersize=4)
    ax1.set_title('Cross Entropy Loss Convergence', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Iterations', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.legend()

    # 2. PERPLEXITY (The "Clinical Certainty" Metric)
    # Perplexity = exp(loss)
    train_ppl = [np.exp(l) for l in train_losses]
    val_ppl = [np.exp(l) for l in val_losses]

    ax2.plot(iters, train_ppl, label='Train PPL', color='#1abc9c', linestyle='--')
    ax2.plot(iters, val_ppl, label='Val PPL', color='#2ecc71', linewidth=2)
    ax2.set_title('Model Perplexity ($e^{loss}$)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Iterations', fontsize=12)
    ax2.set_ylabel('Perplexity Score', fontsize=12)
    ax2.legend()

    # 3. LEARNING RATE SCHEDULE
    # The lr_list contains LR for every iteration; we slice it to match evals
    # or plot the whole list for a smoother curve
    ax3.plot(learning_rates, color='#f39c12', linewidth=1.5)
    ax3.set_title('Cosine Decay LR Schedule', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Iterations', fontsize=12)
    ax3.set_ylabel('Learning Rate', fontsize=12)
    ax3.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

    # Add annotations for best performance
    best_val_idx = np.argmin(val_losses)
    best_iter = iters[best_val_idx]
    best_val = val_losses[best_val_idx]

    ax1.annotate(f'Best: {best_val:.4f}',
                 xy=(best_iter, best_val),
                 xytext=(best_iter, best_val + 0.2),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

    plt.tight_layout()
    plt.savefig("VitalLM_SFT_Analysis.png", dpi=300)
    plt.show()

# Execute plotting
if val_loss_list:
    plot_sft_performance(train_loss_list, val_loss_list, lr_list, eval_interval)
else:
    print("Metrics lists are empty. Ensure your training loop is appending to them!")